In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

print("🚨 Starting Model 2 (V2) Training: Deficit Risk Predictor")

# ==========================================
# 1. GENERATE MOCK DATA (Simulating your DB)
# ==========================================
np.random.seed(42)
num_records = 5000

# Mocking data extracted from Users, Pockets (Variable only), and Daily Spends
data = {
    'monthly_income': np.random.choice([2500, 3500, 5000, 8000], num_records),
    'variable_balance': np.random.randint(50, 3000, num_records), # Sum of VARIABLE pockets
    'days_left_in_month': np.random.randint(1, 30, num_records), 
    'daily_spend_avg': np.random.randint(10, 200, num_records) # Avg from Daily Total Spends
}
df = pd.DataFrame(data)

# ==========================================
# 2. FEATURE ENGINEERING (The Secret Sauce)
# ==========================================
# AI needs to know the "Safe Limit" to understand if the "Daily Spend" is bad.
df['safe_daily_limit'] = df['variable_balance'] / df['days_left_in_month']

# Burn Rate > 1.0 means they are spending faster than they should!
df['burn_rate'] = df['daily_spend_avg'] / df['safe_daily_limit']

# Target Variable (1 = Will go broke, 0 = Safe)
def calculate_actual_risk(row):
    # Projected total spend for the rest of the month
    projected_spend = row['daily_spend_avg'] * row['days_left_in_month']
    
    # We add a 10% buffer. If projected spend exceeds balance by more than 10%, it's high risk.
    if projected_spend > (row['variable_balance'] * 1.10):
        return 1 # HIGH RISK (Deficit expected)
    else:
        return 0 # SAFE

df['is_at_risk'] = df.apply(calculate_actual_risk, axis=1)

print("\n📊 Sample Features fed to the AI:")
print(df[['variable_balance', 'daily_spend_avg', 'safe_daily_limit', 'burn_rate', 'is_at_risk']].head())

# ==========================================
# 3. TRAIN THE RANDOM FOREST CLASSIFIER
# ==========================================
# We feed the model BOTH the raw DB columns and our engineered Burn Rate
X = df[['monthly_income', 'variable_balance', 'days_left_in_month', 'daily_spend_avg', 'burn_rate']]
y = df['is_at_risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n🧠 Training Random Forest Classifier...")
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Check Accuracy
predictions = model.predict(X_test)
print(f"\n✅ Model Trained Successfully!")
print(classification_report(y_test, predictions))

# ==========================================
# 4. SAVE THE MODEL
# ==========================================
joblib.dump(model, 'risk_predictor_v2.pkl')
print("💾 Model saved successfully as 'risk_predictor_v2.pkl'!")

🚨 Starting Model 2 (V2) Training: Deficit Risk Predictor

📊 Sample Features fed to the AI:
   variable_balance  daily_spend_avg  safe_daily_limit  burn_rate  is_at_risk
0              2020               39        118.823529   0.328218           0
1              1760              181        880.000000   0.205682           0
2              2096               21         87.333333   0.240458           0
3              2869               60       2869.000000   0.020913           0
4               952              130        136.000000   0.955882           0

🧠 Training Random Forest Classifier...

✅ Model Trained Successfully!
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       564
           1       1.00      1.00      1.00       436

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

💾 Model saved successfully as 'risk_predict